# Predição de risco de crédito e inadimplência

Experimento acadêmico de classificação binária: **1 = inadimplência; 0 = adimplência**. Comparamos um baseline ingênuo, regressão logística, árvore de decisão, Random Forest e gradient boosting. Não pressupomos que um modelo complexo será melhor. **Não requer GPU nem TensorFlow.**

A fonte é o [Credit Risk Dataset de Lao Tse](https://www.kaggle.com/datasets/laotse/credit-risk-dataset), versão 1, CC0. O autor descreve dados **simulados**, não uma carteira real auditada. Os resultados não autorizam decisões reais de crédito nem definem a honestidade de uma pessoa.

Execute de cima para baixo. No Colab, use CPU e as bibliotecas disponíveis. Para reproduzir o ambiente de referência, instale as versões de `requirements.txt` antes de importar bibliotecas; se alterar pacotes já carregados no Colab, reinicie a sessão. As versões efetivamente usadas serão registradas nos resultados.

In [ ]:
from hashlib import sha256
from importlib.metadata import version
from io import BytesIO
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen
from zipfile import BadZipFile, ZipFile
import json
import platform

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from threadpoolctl import threadpool_limits

SEED = 42
DATASET_URL = 'https://www.kaggle.com/api/v1/datasets/download/laotse/credit-risk-dataset?datasetVersionNumber=1'
DATASET_SHA256 = 'ce3c6d2167717bf1627d1c0c81cbccd28323cd4aa7b96d542599366d5ff6aac8'
NUMERIC_FEATURES = [
    'person_age', 'person_income', 'person_emp_length', 'loan_amnt',
    'loan_percent_income', 'cb_person_cred_hist_length',
]
CATEGORICAL_FEATURES = ['person_home_ownership', 'loan_intent', 'cb_person_default_on_file']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
EXCLUDED_FEATURES = ['loan_grade', 'loan_int_rate']
EXPECTED_COLUMNS = FEATURES + EXCLUDED_FEATURES + ['loan_status']


## 1. Fonte, integridade e limpeza documentada

Baixamos somente a versão 1 da base, sem substituir silenciosamente por outro dataset. O SHA-256 verifica os **bytes do CSV**, inclusive quando carregado manualmente. Caso o Kaggle bloqueie o download, baixe `credit_risk_dataset.csv` na página da fonte e configure `DATA_PATH` abaixo. Não é necessário colocar credenciais no notebook.

Regras fixas, definidas antes de observar resultados: remover duplicatas completas; excluir idades fora de 18–100 anos, renda/valor do empréstimo não positivos, razões de renda negativas e durações negativas ou superiores à idade menos 14. O limite de 14 anos é uma hipótese de plausibilidade, não uma regra universal de trabalho/crédito. Não eliminamos rendas altas por percentis. Ausentes dos preditores permanecem para imputação **somente no treino**. Alvo ausente/inválido e esquema incorreto interrompem a execução.

In [ ]:
def load_data(local_path=None):
    if local_path is not None:
        payload = Path(local_path).read_bytes()
    else:
        try:
            with urlopen(DATASET_URL, timeout=60) as response:
                archive = response.read()
            with ZipFile(BytesIO(archive)) as zipped:
                payload = zipped.read('credit_risk_dataset.csv')
        except (HTTPError, URLError, TimeoutError, BadZipFile, KeyError) as error:
            raise RuntimeError(
                'Falha ao baixar a versão 1 do Kaggle. Baixe o CSV original e configure DATA_PATH; '
                'não será usada outra base como fallback.'
            ) from error
    digest = sha256(payload).hexdigest()
    if digest != DATASET_SHA256:
        raise ValueError('SHA-256 diferente da versão 1 esperada; confira a origem do CSV.')
    return pd.read_csv(BytesIO(payload)), digest


def clean_data(raw):
    missing = sorted(set(EXPECTED_COLUMNS) - set(raw.columns))
    if missing:
        raise ValueError(f'Colunas obrigatórias ausentes: {missing}')
    data = raw[EXPECTED_COLUMNS].copy()
    target = data['loan_status']
    if target.isna().any() or not target.isin([0, 1]).all() or target.nunique() != 2:
        raise ValueError('loan_status deve conter as duas classes 0 e 1, sem ausentes.')
    numeric_columns = NUMERIC_FEATURES + ['loan_int_rate']
    data[numeric_columns] = data[numeric_columns].apply(pd.to_numeric, errors='raise')
    if np.isinf(data[numeric_columns].to_numpy(dtype=float)).any():
        raise ValueError('Atributos numéricos não podem conter infinito.')
    data['loan_status'] = target.astype(int)
    duplicates = int(data.duplicated().sum())
    data = data.drop_duplicates().copy()
    age = data['person_age']
    invalid = (
        (age.notna() & ~age.between(18, 100))
        | (data['person_income'] <= 0) | (data['loan_amnt'] <= 0)
        | (data['loan_percent_income'] < 0)
        | (data['person_emp_length'] < 0) | (data['person_emp_length'] > age - 14)
        | (data['cb_person_cred_hist_length'] < 0)
        | (data['cb_person_cred_hist_length'] > age - 14)
    )
    audit = {
        'rows_raw': len(raw), 'duplicates_removed': duplicates,
        'invalid_rows_removed': int(invalid.sum()),
    }
    data = data.loc[~invalid].copy()
    if data['loan_status'].nunique() != 2:
        raise ValueError('loan_status precisa manter as duas classes após a limpeza.')
    audit['rows_clean'] = len(data)
    return data, audit


def split_data(data):
    X, y = data[FEATURES], data['loan_status']
    X_development, X_test, y_development, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED,
    )
    X_train, X_validation, y_train, y_validation = train_test_split(
        X_development, y_development, test_size=0.25,
        stratify=y_development, random_state=SEED,
    )
    return {
        'train': (X_train, y_train),
        'validation': (X_validation, y_validation),
        'test': (X_test, y_test),
    }


In [ ]:
DATA_PATH = None  # Ex.: '/content/credit_risk_dataset.csv', sempre a versão 1 original.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
sns.set_theme(style='whitegrid')
versions = {'python': platform.python_version()}
versions.update({name: version(name) for name in [
    'numpy', 'pandas', 'scikit-learn', 'matplotlib', 'seaborn', 'scipy', 'threadpoolctl',
]})
raw, data_digest = load_data(DATA_PATH)
data, cleaning_audit = clean_data(raw)
splits = split_data(data)
X_train, y_train = splits['train']
X_validation, y_validation = splits['validation']
# O teste está reservado; não será usado na exploração ou escolha de modelo/limiar.
print('Versões:', versions)
print('Limpeza:', cleaning_audit)
print('Tamanhos:', {name: len(X) for name, (X, _) in splits.items()})
print('SHA-256:', data_digest)


## 2. Exploração somente no treino e disponibilidade dos atributos

Usamos idade, renda anual, tempo de emprego, valor solicitado, razão empréstimo/renda, tempo de histórico de crédito, moradia, finalidade e inadimplência anterior. `cb_person_default_on_file` é o **histórico anterior**, não o alvo do empréstimo atual.

**Excluímos `loan_grade` e `loan_int_rate`**: podem codificar uma decisão prévia da instituição. Sem a data de origem dos campos, não é possível assegurar sua disponibilidade antes da concessão. A exclusão é conservadora e não prova ausência de outros vazamentos. A base não identifica pessoas nem datas; portanto, a separação é por registros, não por cliente ou por período.

O autor não documenta moeda, prazo/horizonte de inadimplência ou mecanismo de simulação. Não atribuímos valores em reais nem interpretamos `1` como recusa de crédito.

In [ ]:
display(X_train.describe(include='all').T)
display(X_train.isna().sum().rename('ausentes_no_treino').to_frame())
print('Proporção de inadimplência no treino:', round(y_train.mean(), 4))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
y_train.value_counts().sort_index().plot.bar(ax=axes[0], color=['#26756e', '#bc5546'])
axes[0].set(title='Classes no treino', xlabel='loan_status', ylabel='Registros')
sns.histplot(np.log1p(X_train['person_income']), bins=40, ax=axes[1])
axes[1].set(title='Renda anual no treino', xlabel='log(1 + renda)')
sns.boxplot(x=y_train, y=X_train['loan_percent_income'], ax=axes[2])
axes[2].set(title='Comprometimento de renda no treino', xlabel='loan_status')
fig.tight_layout()
fig.savefig(RESULTS / '01_exploration.png', dpi=140, bbox_inches='tight')
plt.show()


## 3. Pipelines e comparação de modelos

- **Dummy (prior)**: ignora o perfil e devolve a proporção de classes do treino. Com limiar 0,5, prevê a classe majoritária; ao otimizar F1, pode prever todos como inadimplentes. Esse contraste mostra por que acurácia isolada engana.
- **Regressão logística**: baseline linear regularizado e barato; precisa de escala numérica compatível.
- **Árvore de decisão**: regras não lineares, limitada a profundidade 5 para facilitar interpretação e controlar sobreajuste.
- **Random Forest**: média de 200 árvores, reduz variância; folhas com pelo menos dois registros.
- **HistGradientBoosting**: árvores sequenciais que corrigem erros anteriores; implementação eficiente para dezenas de milhares de linhas, sem dependências de XGBoost/LightGBM.

A imputação por mediana/moda e o vocabulário do One-Hot Encoder são ajustados dentro de cada `Pipeline`, **apenas no treino**. Só a regressão logística usa `StandardScaler`; árvores não precisam de z-score. Categorias novas são aceitas com `handle_unknown='ignore'`. Os hiperparâmetros abaixo são fixos: não houve busca ampla por configurações.

Não aplicamos SMOTE nem balanceamos validação/teste: preservamos a prevalência original. Tratamos o desequilíbrio pela avaliação da classe 1 e pelo limiar escolhido na validação. Scores de `predict_proba` **não são probabilidades calibradas**.

Referências: [prevenção de vazamento](https://scikit-learn.org/1.6/common_pitfalls.html#data-leakage), [OneHotEncoder](https://scikit-learn.org/1.6/modules/generated/sklearn.preprocessing.OneHotEncoder.html), [HistGradientBoosting](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.HistGradientBoostingClassifier.html).

In [ ]:
def make_preprocessor(scale=False):
    numeric_steps = [('imputer', SimpleImputer(strategy='median', keep_empty_features=True))]
    if scale:
        numeric_steps.append(('scaler', StandardScaler()))
    categorical = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ])
    return ColumnTransformer([
        ('numeric', Pipeline(numeric_steps), NUMERIC_FEATURES),
        ('categorical', categorical, CATEGORICAL_FEATURES),
    ], remainder='drop')


def make_models():
    estimators = {
        'Dummy (prior)': DummyClassifier(strategy='prior'),
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=SEED),
        'Decision Tree': DecisionTreeClassifier(
            max_depth=5, min_samples_leaf=30, random_state=SEED,
        ),
        'Random Forest': RandomForestClassifier(
            n_estimators=200, min_samples_leaf=2, random_state=SEED, n_jobs=2,
        ),
        'HistGradientBoosting': HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.08, max_leaf_nodes=15,
            l2_regularization=1.0, early_stopping=False, random_state=SEED,
        ),
    }
    return {
        name: Pipeline([
            ('preprocessor', make_preprocessor(scale=name == 'Logistic Regression')),
            ('classifier', estimator),
        ])
        for name, estimator in estimators.items()
    }


def choose_threshold(y_validation, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_validation, probabilities)
    denominator = precision[:-1] + recall[:-1]
    scores = np.divide(
        2 * precision[:-1] * recall[:-1], denominator,
        out=np.zeros_like(denominator), where=denominator > 0,
    )
    # Empate: menor limiar, favorecendo recall. Não usar y_test nesta função.
    return float(thresholds[np.argmax(scores)])


def evaluate(target, probabilities, threshold):
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)
    return {
        'threshold': float(threshold),
        'accuracy': float(accuracy_score(target, predictions)),
        'precision': float(precision_score(target, predictions, zero_division=0)),
        'recall': float(recall_score(target, predictions, zero_division=0)),
        'f1': float(f1_score(target, predictions, zero_division=0)),
        'roc_auc': float(roc_auc_score(target, probabilities)),
        'average_precision': float(average_precision_score(target, probabilities)),
        'confusion_matrix': confusion_matrix(target, predictions, labels=[0, 1]).tolist(),
    }


## 4. Escolha do modelo e do limiar — somente validação

Cada modelo é treinado com os mesmos 60% dos registros. Os 20% de validação escolhem o limiar que maximiza o **F1 da classe 1**. Escolhemos o modelo por esse F1; em empate, maior Average Precision e depois nome. Nenhum ajuste usa os 20% de teste.

**F1 não minimiza prejuízo financeiro**: é uma convenção acadêmica sem custos monetários conhecidos. Mostramos também F1 no limiar 0,5, precision, recall, ROC-AUC e Average Precision (resumo da curva PR, não a integral trapezoidal). A validação fica otimista após selecionar modelo e limiar nela; somente o teste reservado fornece a avaliação final deste experimento.

Referência: [ajuste do limiar de decisão](https://scikit-learn.org/1.6/modules/classification_threshold.html).

In [ ]:
models = make_models()
validation_metrics, thresholds, validation_scores = {}, {}, {}
with threadpool_limits(limits=2):
    for name, model in models.items():
        model.fit(X_train, y_train)
        probabilities = model.predict_proba(X_validation)[:, 1]
        threshold = choose_threshold(y_validation, probabilities)
        thresholds[name] = threshold
        validation_scores[name] = probabilities
        validation_metrics[name] = evaluate(y_validation, probabilities, threshold)
        validation_metrics[name]['f1_at_0_5'] = float(f1_score(
            y_validation, probabilities >= 0.5, zero_division=0,
        ))
        train_probabilities = model.predict_proba(X_train)[:, 1]
        validation_metrics[name]['train_f1'] = float(f1_score(
            y_train, train_probabilities >= threshold, zero_division=0,
        ))
validation_table = pd.DataFrame(validation_metrics).T
validation_table.index.name = 'model'
validation_table = validation_table.reset_index().sort_values(
    ['f1', 'average_precision', 'model'], ascending=[False, False, True],
).set_index('model')
best_name = validation_table.index[0]
best_model, best_threshold = models[best_name], thresholds[best_name]
display(validation_table.drop(columns='confusion_matrix').astype(float).round(4))
print(f'Escolhido na validação: {best_name}; limiar = {best_threshold:.6f}')
# Congelamos modelos e limiares. Não retreinamos em treino+validação, pois isso mudaria os scores.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
validation_table[['f1_at_0_5', 'f1', 'average_precision']].astype(float).plot.barh(ax=axes[0])
axes[0].set(title='Comparação na validação', xlabel='Score', xlim=(0, 1))
precision, recall, candidates = precision_recall_curve(y_validation, validation_scores[best_name])
f1_curve = np.divide(
    2 * precision[:-1] * recall[:-1], precision[:-1] + recall[:-1],
    out=np.zeros_like(candidates), where=(precision[:-1] + recall[:-1]) > 0,
)
axes[1].plot(candidates, precision[:-1], label='Precision')
axes[1].plot(candidates, recall[:-1], label='Recall')
axes[1].plot(candidates, f1_curve, label='F1')
axes[1].axvline(best_threshold, color='black', linestyle='--', label='Limiar escolhido')
axes[1].set(title=f'{best_name} — validação', xlabel='Limiar', ylabel='Score', ylim=(0, 1.05))
axes[1].legend()
fig.tight_layout()
fig.savefig(RESULTS / '02_validation.png', dpi=140, bbox_inches='tight')
plt.show()


## 5. Inspeção exploratória na validação

A importância por permutação mede a queda de Average Precision ao embaralhar uma coluna original. Usamos até 1.000 registros da validação e três repetições para manter o custo baixo. Correlação entre atributos pode diluir a importância: **não é causalidade nem justificativa individual de recusa**. Esta inspeção não altera a lista de atributos ou o modelo nesta execução.

In [ ]:
X_inspection = X_validation.sample(n=min(1000, len(X_validation)), random_state=SEED)
y_inspection = y_validation.loc[X_inspection.index]
with threadpool_limits(limits=2):
    importance = permutation_importance(
        best_model, X_inspection, y_inspection, scoring='average_precision',
        n_repeats=3, random_state=SEED, n_jobs=1,
    )
importance_table = pd.DataFrame({
    'feature': FEATURES, 'mean': importance.importances_mean, 'std': importance.importances_std,
}).sort_values('mean')
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(importance_table['feature'], importance_table['mean'], xerr=importance_table['std'])
ax.set(title=f'Importância por permutação — {best_name}', xlabel='Queda de Average Precision (validação)')
fig.tight_layout()
fig.savefig(RESULTS / '03_importance.png', dpi=140, bbox_inches='tight')
plt.show()


## 6. Avaliação final no teste reservado

Só agora acessamos os rótulos do teste para medir desempenho. Todos os modelos/limiares já estão congelados. A tabela segue a **ordem escolhida na validação**, não um ranking usado para escolher outro vencedor no teste. Comparamos também o limiar padrão 0,5 do modelo selecionado.

Na matriz: falso negativo = inadimplência não detectada; falso positivo = adimplência sinalizada como risco. Os erros não significam aprovação/recusa automática. Não retune o modelo após ver estes resultados sem reservar um novo teste.

In [ ]:
X_test, y_test = splits['test']
test_metrics, test_scores = {}, {}
with threadpool_limits(limits=2):
    for name in validation_table.index:
        probabilities = models[name].predict_proba(X_test)[:, 1]
        test_scores[name] = probabilities
        test_metrics[name] = evaluate(y_test, probabilities, thresholds[name])
test_table = pd.DataFrame(test_metrics).T
display(test_table.drop(columns='confusion_matrix').astype(float).round(4))
best_probabilities = test_scores[best_name]
best_predictions = (best_probabilities >= best_threshold).astype(int)
selected_at_0_5 = evaluate(y_test, best_probabilities, 0.5)
print('Modelo selecionado na validação:', best_name)
print(classification_report(
    y_test, best_predictions, labels=[0, 1],
    target_names=['Adimplente', 'Inadimplente'], zero_division=0,
))
display(pd.DataFrame({
    'limiar_0.5': selected_at_0_5, 'limiar_validacao': test_metrics[best_name],
}).drop(index='confusion_matrix').astype(float).round(4))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, labels=[0, 1], display_labels=['Adimplente', 'Inadimplente'],
    ax=axes[0], colorbar=False, cmap='Blues',
)
axes[0].set_title('Matriz de confusão — teste')
RocCurveDisplay.from_predictions(y_test, best_probabilities, ax=axes[1], name=best_name)
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].set_title('ROC — teste')
PrecisionRecallDisplay.from_predictions(y_test, best_probabilities, ax=axes[2], name=best_name)
axes[2].axhline(y_test.mean(), linestyle='--', color='gray', label='Prevalência')
axes[2].set_title('Precision–Recall — teste')
axes[2].legend()
fig.tight_layout()
fig.savefig(RESULTS / '04_test_evaluation.png', dpi=140, bbox_inches='tight')
plt.show()


In [ ]:
results = {
    'dataset': {'url': DATASET_URL, 'version': 1, 'csv_sha256': data_digest, 'simulated': True},
    'seed': SEED, 'versions': versions, 'cleaning': cleaning_audit,
    'features': FEATURES, 'excluded_features': EXCLUDED_FEATURES,
    'splits': {name: {
        'rows': len(X), 'defaults': int(y.sum()), 'default_rate': float(y.mean()),
    } for name, (X, y) in splits.items()},
    'parameters': {name: model.named_steps['classifier'].get_params() for name, model in models.items()},
    'selection': {'criterion': 'validation_f1_class_1', 'model': best_name, 'threshold': best_threshold},
    'validation': validation_metrics, 'test': test_metrics,
    'selected_test_at_0_5': selected_at_0_5,
    'permutation_importance_validation': importance_table.to_dict(orient='records'),
}
(RESULTS / 'metrics.json').write_text(
    json.dumps(results, indent=2, ensure_ascii=False, allow_nan=False) + '\n', encoding='utf-8',
)
print('Métricas agregadas e quatro figuras salvas em:', RESULTS)
print('Nenhum dado individual, credencial ou modelo serializado foi exportado.')


## 7. Limitações, ética e próximos experimentos

- Uma única divisão aleatória e uma base simulada não demonstram generalização para clientes reais, outros períodos ou o Brasil. Não há intervalo de confiança, validação temporal, auditoria por pessoa ou garantia de estabilidade.
- Dados históricos/simulados podem reproduzir discriminação por renda, idade ou moradia. Excluir nomes de atributos sensíveis não elimina seus proxies. Este projeto **não conclui que o modelo é justo**; faltam atributos e contexto para uma auditoria completa.
- Antes de qualquer uso real: base legítima sob LGPD, critérios legais, calibração de probabilidades, avaliação de subgrupos, monitoramento de drift, revisão humana e possibilidade de contestação. O score não representa caráter ou honestidade.
- Melhorias possíveis: validação cruzada estratificada **no desenvolvimento**, ajuste moderado de hiperparâmetros, CatBoost para categóricas e XGBoost/LightGBM para boosting. Cada nova comparação requer preservar o teste; não há motivo para impor Deep Learning sem evidência de ganho.
- AP resume a curva Precision–Recall. Referências: [Average Precision](https://scikit-learn.org/1.6/modules/generated/sklearn.metrics.average_precision_score.html) e [importância por permutação](https://scikit-learn.org/1.6/modules/permutation_importance.html).